In [4]:
import torch
import torchvision.models as models
import time
import os

## Baseline inference

In [6]:
def benchmark(model, dummy_input=None, num_runs=100):
    if dummy_input is None:
        dummy_input = torch.randn(1, 3, 224, 224)
    
    start = time.time()
    # Warmup
    for _ in range(3):
        with torch.no_grad():
            model(dummy_input)
    end = time.time()
    warmup_time = (end - start)/3

    # Benchmark
    start = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            model(dummy_input)
    end = time.time()

    avg_time = (end - start) / num_runs
    return warmup_time, avg_time

In [7]:
model_fp32 = models.resnet18(pretrained=True).eval()
dummy_input = torch.randn(1, 3, 224, 224)

# Benchmark the FP32 model
warmup_time, avg_time = benchmark(model_fp32, dummy_input)

print(f"Warmup Time: {warmup_time:.4f} seconds")
print(f"Avg Inference Time: {avg_time:.4f} seconds")

# Benchmark and save
torch.save(model_fp32.state_dict(), "fp32.pth")
print("FP32 size (MB):", os.path.getsize("fp32.pth") / 1e6)

Warmup Time: 0.0233 seconds
Avg Inference Time: 0.0219 seconds
FP32 size (MB): 46.828292


## Pruning

In [5]:
# TODO

## Quntization

In [8]:
from torch.quantization import quantize_dynamic

model_dynamic = quantize_dynamic(model_fp32, {torch.nn.Linear}, dtype=torch.qint8)


warmup_time, avg_time = benchmark(model_dynamic, dummy_input)
print(f"Warmup Time: {warmup_time:.4f} seconds")
print(f"Avg Inference Time: {avg_time:.4f} seconds")

torch.save(model_dynamic.state_dict(), "dynamic.pth")
print("Dynamic Quantized size (MB):", os.path.getsize("dynamic.pth") / 1e6)
# Benchmark the dynamic quantized model


Warmup Time: 0.0247 seconds
Avg Inference Time: 0.0216 seconds
Dynamic Quantized size (MB): 45.299578


In [ ]:
# TODO